# 07 · Why skills exist

## Goal

Measure, side by side, what instructions cost every single turn versus what
a skill costs only when selected. Export one skill, import it into a second
throwaway agent, and see that a skill is a portable markdown artefact, not
an agent-specific setting.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)


## Concept

Instructions are always in the context window and always paid for, on
every single turn, whether or not that turn needs them. A skill — a
markdown file with a description the selector matches against — is only
loaded into context when the agent decides it's relevant to the current
turn. For anything used occasionally (a specific negotiation playbook, a
rarely-needed escalation procedure), that's the difference between paying
for it once per conversation and paying for it on every message.

Skills are markdown and fully portable: nothing about a `SKILL.md` file
ties it to one agent. You'll export the routing skill built in `08` here
first, in miniature, to see the mechanism before building the real bundle.


## Build


### Token accounting: instructions-only vs. skill-gated


In [ ]:
import tiktoken  # or your model's tokenizer of choice
enc = tiktoken.get_encoding("cl100k_base")

instructions_text = open("../agents/contract-renewal-desk/instructions.md").read()
instructions_tokens = len(enc.encode(instructions_text))

negotiation_playbook = '''
## Negotiation playbook (rarely needed — only on explicit escalation)
... (a long, detailed playbook, several hundred tokens) ...
'''
playbook_tokens = len(enc.encode(negotiation_playbook))

turns_per_conversation = 8
print(f"instructions: {instructions_tokens} tokens x {turns_per_conversation} turns = {instructions_tokens * turns_per_conversation} tokens/conversation")
print(f"as a skill, loaded ~1 turn in 5 conversations: {playbook_tokens} tokens x 0.2 = {playbook_tokens * 0.2:.0f} tokens/conversation amortised")


### A minimal exportable skill


In [ ]:
from pathlib import Path
skill_dir = Path("../skills/negotiation-playbook")
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text('''---
name: negotiation-playbook
description: Use when drafting a renewal negotiation position after spend/performance signals indicate escalation is warranted.
---

# Negotiation playbook

1. Lead with the performance data, not a demand.
2. Anchor on the contract's existing renewal clause, not a fresh ask.
3. Never issue a deadline in the first message.
''')
print("skill written — portable markdown, not agent-specific config")


### Import into a second, throwaway agent to prove portability


In [ ]:
from csx.pac import copilot_init, copilot_push
throwaway = Path("../agents/skill-portability-check")
copilot_init(throwaway)
import shutil
(throwaway / "skills").mkdir(exist_ok=True)
shutil.copytree(skill_dir, throwaway / "skills" / "negotiation-playbook", dirs_exist_ok=True)
copilot_push(throwaway)
print("same SKILL.md, second agent — no per-agent rewrite needed")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("07", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="token accounting comparison + portability check (no instructions/knowledge change)")


## Teardown


In [ ]:
import subprocess
subprocess.run(["pac", "copilot", "delete", "--name", "crd_skill-portability-check"], check=False)
print("throwaway portability-check agent deleted; negotiation-playbook skill kept for 08")
